# 09 — Build glacier masks from GLIMS outlines

**Purpose.** Makes the training labels: one binary mask per glacier, burned from its GLIMS outline
(`glacierview.training.rasterize_outline`). Each outline is reprojected and rasterised onto the
glacier's reference grid (`glacierview.rasters.reference_grid`): 30 m UTM over the DEM's extent, the
same grid every scene is resampled onto in training and at inference. 1 is inside the outline and 0
outside. So the mask lines up with whichever scene notebook 10 pairs it with, whatever UTM zone that
scene came in. Notebook 10 resizes it to 128×128.

## Inputs
- **`config.glims_selection_path()`**: GLIMS outlines from notebook 01.
- **`image_attributes.csv`** in `config.processed_metadata_dir(config.TRAINING_DATA_LABEL)`: from
  notebook 06; lists the glaciers that have training scenes.
- **The training DEMs**: `config.dem_dir(config.TRAINING_DATA_LABEL)`, from notebook 04; each one
  defines its glacier's grid.

## Outputs
- `config.masks_dir()/<glims_id>.tif` (`src/segmentation/training/data/masks_reference_grid/`): one
  mask per glacier, on its reference grid. Gitignored, with no S3 copy. The published pipeline's
  masks, each drawn on one scene's grid, stay in `config.published_masks_dir()` (`masks_staging_2/`).

In [ ]:
import pandas as pd
from PIL import Image
from tqdm import tqdm

from glacierview import config, glims, rasters
from glacierview.training import rasterize_outline

LABEL = config.TRAINING_DATA_LABEL
attributes = pd.read_csv(config.processed_metadata_dir(LABEL) / "image_attributes.csv",
                         usecols=["file_name"])
glims_ids = pd.Index(attributes["file_name"].str.split("_").str[0].unique())
outlines = glims.read_selection().set_index("glac_id").geometry

In [ ]:
out = config.masks_dir()
out.mkdir(parents=True, exist_ok=True)
for glims_id in tqdm(glims_ids.intersection(outlines.index)):
    grid = rasters.reference_grid(config.dem_path(glims_id, LABEL))
    Image.fromarray(rasterize_outline(grid, outlines[glims_id])).save(out / f"{glims_id}.tif")